In [ ]:
!pip install glymur scikit-image --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 27.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
Rate-Distortion Validation for JPEG2000 Compression Study
==========================================================
Computes PSNR and SSIM at each compression ratio for BreakHis, BACH, and MHIST.

Run in Google Colab with GPU runtime.
Mount Drive first: drive.mount('/content/drive')

Output:
  - rate_distortion_results.csv  (per-image raw data)
  - rate_distortion_summary.csv  (mean ± SD per dataset × CR, paper Table)
  - rate_distortion_table.tex    (LaTeX table for paper)
"""

# ============================================================
# 1. INSTALL & IMPORTS
# ============================================================
# !pip install glymur scikit-image --quiet

import os
import glob
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import glymur
import tempfile
import time

warnings.filterwarnings('ignore')

# ============================================================
# 2. CONFIGURATION — EDIT PATHS HERE
# ============================================================

# Where your RESIZED 224×224 images live (the ones you used for experiments).
# If you compressed from 224×224 (as the paper states), point to those folders.
# If images are still at original resolution, the script resizes to 224×224.

DATASET_CONFIG = {
    'BreakHis': {
        'image_dir': '/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/images',
        # Adjust ↑ to wherever your BreakHis 224×224 PNGs live.
        # Common alternatives:
        #   '/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/data/all_images'
        #   '/content/drive/MyDrive/BreakHis_Results/images'
        'extensions': ['*.png', '*.PNG'],
        'original_resolution': (700, 460),  # for reference only
        'target_size': (224, 224),
        'max_images': None,  # None = all images; set to 500 for quick test
    },
    'BACH': {
        'image_dir': '/content/drive/MyDrive/BACH_PLOS_ONE/BACH_PLOS_ONE_FINAL/images',
        'extensions': ['*.png', '*.tif', '*.tiff', '*.PNG'],
        'original_resolution': (2048, 1536),
        'target_size': (224, 224),
        'max_images': None,
    },
    'MHIST': {
        'image_dir': '/content/drive/MyDrive/MHIST_PLOS_ONE/MHIST_Results_Final/images',
        'extensions': ['*.png', '*.PNG', '*.jpg'],
        'original_resolution': (224, 224),
        'target_size': (224, 224),
        'max_images': None,
    },
}

COMPRESSION_RATIOS = [15, 25, 50, 100, 500, 1000]

OUTPUT_DIR = '/content/drive/MyDrive/JPEG2000_Paper_Final/rate_distortion'

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def find_images(image_dir, extensions):
    """Recursively find all images matching extensions."""
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(image_dir, '**', ext), recursive=True))
    paths = sorted(set(paths))
    return paths


def load_and_resize(path, target_size=(224, 224)):
    """Load image, convert to RGB, resize to target."""
    img = Image.open(path).convert('RGB')
    if img.size != target_size:
        img = img.resize(target_size, Image.LANCZOS)
    return np.array(img, dtype=np.uint8)


def compress_and_decompress(img_array, compression_ratio):
    """
    JPEG2000 compress at given ratio, then decompress.
    Returns: (decompressed_array, achieved_cr, file_size_bytes)
    """
    with tempfile.NamedTemporaryFile(suffix='.jp2', delete=False) as tmp:
        tmp_path = tmp.name

    try:
        # Write original as temporary PNG for size reference
        original_size = img_array.nbytes  # H × W × 3 bytes

        # Compress with Glymur (same as paper: cratios parameter)
        glymur.Jp2k(tmp_path, data=img_array, cratios=[compression_ratio])

        # Read back compressed file size
        compressed_size = os.path.getsize(tmp_path)

        # Decompress
        decompressed = glymur.Jp2k(tmp_path)[:]

        # Achieved compression ratio
        achieved_cr = original_size / compressed_size

        return decompressed, achieved_cr, compressed_size

    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def compute_metrics(original, compressed):
    """Compute PSNR and SSIM between original and compressed images."""
    psnr = compute_psnr(original, compressed, data_range=255)
    ssim = compute_ssim(
        original, compressed,
        data_range=255,
        channel_axis=2,       # RGB channel axis
        win_size=7,           # 7×7 window (safe for 224×224)
    )
    return psnr, ssim


# ============================================================
# 4. MAIN COMPUTATION LOOP
# ============================================================

def run_rate_distortion(dataset_config, compression_ratios):
    """Run PSNR/SSIM computation for all datasets and CRs."""

    all_records = []

    for dataset_name, cfg in dataset_config.items():
        print(f"\n{'='*60}")
        print(f"  Processing: {dataset_name}")
        print(f"{'='*60}")

        image_dir = cfg['image_dir']

        # Check if directory exists
        if not os.path.isdir(image_dir):
            print(f"  ⚠️  Directory not found: {image_dir}")
            print(f"  → Update DATASET_CONFIG['{dataset_name}']['image_dir']")
            print(f"  → Looking for images in parent directories...")

            # Try to find images in parent
            parent = os.path.dirname(image_dir)
            if os.path.isdir(parent):
                for root, dirs, files in os.walk(parent):
                    img_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.tif', '.tiff'))]
                    if len(img_files) > 10:
                        print(f"  → Found {len(img_files)} images in: {root}")
                        break
            print(f"  SKIPPING {dataset_name}\n")
            continue

        # Find images
        image_paths = find_images(image_dir, cfg['extensions'])

        if cfg.get('max_images'):
            np.random.seed(42)
            indices = np.random.choice(len(image_paths), min(cfg['max_images'], len(image_paths)), replace=False)
            image_paths = [image_paths[i] for i in sorted(indices)]

        print(f"  Found {len(image_paths)} images")
        if len(image_paths) == 0:
            print(f"  ⚠️  No images found! Check path and extensions.")
            continue

        target_size = cfg['target_size']
        t_start = time.time()

        for i, img_path in enumerate(image_paths):
            if (i + 1) % 200 == 0 or i == 0:
                elapsed = time.time() - t_start
                rate = (i + 1) / elapsed if elapsed > 0 else 0
                eta = (len(image_paths) - i - 1) / rate if rate > 0 else 0
                print(f"  [{i+1}/{len(image_paths)}] "
                      f"({rate:.1f} img/s, ETA: {eta/60:.1f} min)")

            try:
                original = load_and_resize(img_path, target_size)
            except Exception as e:
                print(f"  ⚠️  Error loading {img_path}: {e}")
                continue

            for cr in compression_ratios:
                try:
                    decompressed, achieved_cr, file_size = compress_and_decompress(original, cr)
                    psnr, ssim = compute_metrics(original, decompressed)

                    all_records.append({
                        'dataset': dataset_name,
                        'image': os.path.basename(img_path),
                        'target_cr': cr,
                        'achieved_cr': round(achieved_cr, 2),
                        'file_size_bytes': file_size,
                        'psnr_db': round(psnr, 4),
                        'ssim': round(ssim, 6),
                    })

                except Exception as e:
                    print(f"  ⚠️  CR{cr} failed for {os.path.basename(img_path)}: {e}")
                    continue

        elapsed = time.time() - t_start
        n_done = sum(1 for r in all_records if r['dataset'] == dataset_name)
        print(f"  ✅ {dataset_name} done: {n_done} measurements in {elapsed/60:.1f} min")

    return pd.DataFrame(all_records)


# ============================================================
# 5. SUMMARY TABLE & LATEX OUTPUT
# ============================================================

def generate_summary(df, output_dir):
    """Generate summary statistics and LaTeX table."""

    os.makedirs(output_dir, exist_ok=True)

    # --- Raw results ---
    raw_path = os.path.join(output_dir, 'rate_distortion_results.csv')
    df.to_csv(raw_path, index=False)
    print(f"\n📄 Raw results saved: {raw_path}")

    # --- Summary table: mean ± SD per dataset × CR ---
    summary_rows = []

    for dataset in ['BreakHis', 'BACH', 'MHIST']:
        dset = df[df['dataset'] == dataset]
        if dset.empty:
            continue

        for cr in sorted(dset['target_cr'].unique()):
            subset = dset[dset['target_cr'] == cr]

            summary_rows.append({
                'Dataset': dataset,
                'Target CR': f'CR{cr}',
                'N Images': len(subset),
                'Achieved CR (mean)': round(subset['achieved_cr'].mean(), 1),
                'Achieved CR (SD)': round(subset['achieved_cr'].std(), 1),
                'PSNR (mean)': round(subset['psnr_db'].mean(), 2),
                'PSNR (SD)': round(subset['psnr_db'].std(), 2),
                'SSIM (mean)': round(subset['ssim'].mean(), 4),
                'SSIM (SD)': round(subset['ssim'].std(), 4),
            })

    summary = pd.DataFrame(summary_rows)
    summary_path = os.path.join(output_dir, 'rate_distortion_summary.csv')
    summary.to_csv(summary_path, index=False)
    print(f"📄 Summary saved: {summary_path}")

    # --- LaTeX table ---
    latex_lines = []
    latex_lines.append(r'\begin{table}[htbp]')
    latex_lines.append(r'\centering')
    latex_lines.append(r'\caption{Rate-distortion validation: PSNR (dB) and SSIM at each compression ratio.')
    latex_lines.append(r'Values reported as mean $\pm$ SD across all images in each dataset.}')
    latex_lines.append(r'\label{tab:rate_distortion}')
    latex_lines.append(r'\begin{tabular}{llcccc}')
    latex_lines.append(r'\hline')
    latex_lines.append(r'Dataset & CR Level & Achieved CR & PSNR (dB) & SSIM \\')
    latex_lines.append(r'\hline')

    for dataset in ['BreakHis', 'BACH', 'MHIST']:
        ds = summary[summary['Dataset'] == dataset]
        if ds.empty:
            continue

        for idx, row in ds.iterrows():
            cr_label = row['Target CR']
            acr = f"{row['Achieved CR (mean)']:.1f} $\\pm$ {row['Achieved CR (SD)']:.1f}"
            psnr = f"{row['PSNR (mean)']:.2f} $\\pm$ {row['PSNR (SD)']:.2f}"
            ssim = f"{row['SSIM (mean)']:.4f} $\\pm$ {row['SSIM (SD)']:.4f}"

            # Use \multirow for first row of each dataset
            if idx == ds.index[0]:
                n_rows = len(ds)
                latex_lines.append(
                    f"\\multirow{{{n_rows}}}{{*}}{{{dataset}}} & {cr_label} & {acr} & {psnr} & {ssim} \\\\"
                )
            else:
                latex_lines.append(
                    f" & {cr_label} & {acr} & {psnr} & {ssim} \\\\"
                )

        latex_lines.append(r'\hline')

    latex_lines.append(r'\end{tabular}')
    latex_lines.append(r'\end{table}')

    latex_str = '\n'.join(latex_lines)
    latex_path = os.path.join(output_dir, 'rate_distortion_table.tex')
    with open(latex_path, 'w') as f:
        f.write(latex_str)
    print(f"📄 LaTeX table saved: {latex_path}")

    # --- Print preview ---
    print(f"\n{'='*70}")
    print("  RATE-DISTORTION SUMMARY (for paper)")
    print(f"{'='*70}")
    print(summary.to_string(index=False))
    print(f"{'='*70}")

    # --- Key thresholds for paper narrative ---
    print(f"\n📊 KEY THRESHOLDS FOR PAPER:")
    for dataset in ['BreakHis', 'BACH', 'MHIST']:
        ds = summary[summary['Dataset'] == dataset]
        if ds.empty:
            continue
        print(f"\n  {dataset}:")
        for _, row in ds.iterrows():
            flag = ''
            if row['PSNR (mean)'] < 30:
                flag = ' ⚠️  BELOW 30 dB (Chen et al. threshold)'
            elif row['PSNR (mean)'] < 35:
                flag = ' ⚡ Approaching quality threshold'
            print(f"    {row['Target CR']:>6s}: PSNR={row['PSNR (mean)']:6.2f} dB, "
                  f"SSIM={row['SSIM (mean)']:.4f}{flag}")

    return summary


# ============================================================
# 6. RUN EVERYTHING
# ============================================================

if __name__ == '__main__':
    print("=" * 60)
    print("  JPEG2000 Rate-Distortion Validation")
    print("  PSNR & SSIM at CR15–CR1000 for 3 datasets")
    print("=" * 60)

    # Step 1: Mount Drive (if in Colab)
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print("Not in Colab — assuming local paths.")

    # Step 2: Verify paths exist
    print("\n🔍 Checking dataset paths...")
    for name, cfg in DATASET_CONFIG.items():
        exists = os.path.isdir(cfg['image_dir'])
        status = '✅' if exists else '❌'
        print(f"  {status} {name}: {cfg['image_dir']}")
        if not exists:
            print(f"     → FIX THIS PATH before running!")

    # Step 3: Run computation
    print("\n🚀 Starting PSNR/SSIM computation...")
    df = run_rate_distortion(DATASET_CONFIG, COMPRESSION_RATIOS)

    if df.empty:
        print("\n❌ No results! Check your image paths above.")
    else:
        # Step 4: Generate summary and LaTeX
        summary = generate_summary(df, OUTPUT_DIR)

        print(f"\n✅ DONE! {len(df)} total measurements across {df['dataset'].nunique()} datasets.")
        print(f"   Results in: {OUTPUT_DIR}")
        print(f"\n   Next: Copy rate_distortion_table.tex into your paper's §4.1")

  JPEG2000 Rate-Distortion Validation
  PSNR & SSIM at CR15–CR1000 for 3 datasets
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🔍 Checking dataset paths...
  ❌ BreakHis: /content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/images
     → FIX THIS PATH before running!
  ❌ BACH: /content/drive/MyDrive/BACH_PLOS_ONE/BACH_PLOS_ONE_FINAL/images
     → FIX THIS PATH before running!
  ❌ MHIST: /content/drive/MyDrive/MHIST_PLOS_ONE/MHIST_Results_Final/images
     → FIX THIS PATH before running!

🚀 Starting PSNR/SSIM computation...

  Processing: BreakHis
  ⚠️  Directory not found: /content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/images
  → Update DATASET_CONFIG['BreakHis']['image_dir']
  → Looking for images in parent directories...
  SKIPPING BreakHis


  Processing: BACH
  ⚠️  Directory not found: /content/drive/MyDrive/BACH_PLOS_ONE/BACH_PLOS_ONE_FINAL/images
  → Update DATASET_CONFIG['BACH']['image_dir']
  → Looking

In [ ]:
"""
Rate-Distortion Metrics from Existing Compressed Folders
=========================================================
You already have: CR1_original/, CR15/, CR25/, CR50/, CR100/, CR500/, CR1000/
This script pairs originals with compressed versions and computes PSNR + SSIM.

Works for MHIST, BACH, NCT-CRC, or any dataset with this folder structure.
"""

# Cell 1: Install
# !pip install scikit-image --quiet

# Cell 2: Imports
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import time
import warnings
import concurrent.futures
from functools import partial

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION — EDIT THESE PATHS
# ============================================================

DATASETS = {
    'MHIST': {
        'base_dir': '/content/drive/MyDrive/MHIST_PLOS_ONE/MHIST_Split',
        'original_folder': 'CR1_original',  # folder name for originals
        'cr_folders': ['CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
    },
    'BACH': {
        'base_dir': '/content/drive/MyDrive/BACH_PLOS_ONE/BACH_Split',  # UPDATE THIS
        'original_folder': 'CR1_original',
        'cr_folders': ['CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
    },
    'NCT-CRC': {
        'base_dir': '/content/drive/MyDrive/NCT_CRC_PLOS_ONE/NCT_CRC_Split',
        'original_folder': 'CR1_original',
        'cr_folders': ['CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
    },
}

OUTPUT_DIR = '/content/drive/MyDrive/JPEG2000_Paper_Final/rate_distortion'

# Set to a number (e.g., 1000) for quick test, None for full run
MAX_IMAGES_PER_DATASET = None

# ============================================================
# CORE FUNCTION
# ============================================================

def process_single_image(orig_path, orig_dir, base_dir, cr_folders, dataset_name):
    records = []
    try:
        orig_img = np.array(Image.open(orig_path).convert('RGB'))
    except Exception as e:
        return records

    rel_path = os.path.relpath(orig_path, orig_dir)
    filename = os.path.basename(orig_path)

    orig_size = os.path.getsize(orig_path)
    h, w = orig_img.shape[:2]

    for cr_folder in cr_folders:
        cr_value = int(cr_folder.replace('CR', ''))
        compressed_path = os.path.join(base_dir, cr_folder, rel_path)

        if not os.path.exists(compressed_path):
            candidates = glob.glob(os.path.join(base_dir, cr_folder, '**', filename), recursive=True)
            if candidates:
                compressed_path = candidates[0]
            else:
                stem = os.path.splitext(filename)[0]
                found = False
                for ext in ['.png', '.PNG', '.jpg', '.JPG']:
                    candidates = glob.glob(os.path.join(base_dir, cr_folder, '**', stem + ext), recursive=True)
                    if candidates:
                        compressed_path = candidates[0]
                        found = True
                        break
                if not found:
                    continue

        try:
            comp_img = np.array(Image.open(compressed_path).convert('RGB'))
        except Exception:
            continue

        if orig_img.shape != comp_img.shape:
            comp_img = np.array(Image.fromarray(comp_img).resize((orig_img.shape[1], orig_img.shape[0]), Image.LANCZOS))

        psnr = compute_psnr(orig_img, comp_img, data_range=255)
        ssim = compute_ssim(
            orig_img, comp_img,
            data_range=255,
            channel_axis=2,
            win_size=min(7, min(orig_img.shape[0], orig_img.shape[1]))
        )

        comp_size = os.path.getsize(compressed_path)
        achieved_cr = orig_size / comp_size if comp_size > 0 else 0
        bpp = (comp_size * 8) / (h * w) if (h * w) > 0 else 0

        records.append({
            'dataset': dataset_name,
            'filename': filename,
            'compression_ratio': cr_value,
            'achieved_cr': round(achieved_cr, 4),
            'compressed_bytes': comp_size,
            'bpp': round(bpp, 6),
            'psnr_db': round(psnr, 4),
            'ssim': round(ssim, 6),
        })

    return records

def compute_rd_metrics(base_dir, original_folder, cr_folders, dataset_name, max_images=None):
    orig_dir = os.path.join(base_dir, original_folder)

    if not os.path.isdir(orig_dir):
        print(f"  ❌ Original folder not found: {orig_dir}")
        return pd.DataFrame()

    extensions = ('*.png', '*.PNG', '*.jpg', '*.JPG', '*.jpeg', '*.tif', '*.tiff')
    orig_paths = []
    for ext in extensions:
        orig_paths.extend(glob.glob(os.path.join(orig_dir, '**', ext), recursive=True))
    orig_paths = sorted(set(orig_paths))

    print(f"  Found {len(orig_paths)} original images in {orig_dir}")

    if max_images and len(orig_paths) > max_images:
        np.random.seed(42)
        idx = np.random.choice(len(orig_paths), max_images, replace=False)
        orig_paths = [orig_paths[i] for i in sorted(idx)]
        print(f"  Subsampled to {max_images} images for speed")

    records = []
    t0 = time.time()

    worker = partial(process_single_image, orig_dir=orig_dir, base_dir=base_dir, cr_folders=cr_folders, dataset_name=dataset_name)

    with concurrent.futures.ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        futures = {executor.submit(worker, path): path for path in orig_paths}

        for i, future in enumerate(concurrent.futures.as_completed(futures)):
            if (i + 1) % 100 == 0 or i == 0:
                elapsed = time.time() - t0
                rate = (i + 1) / elapsed if elapsed > 0 else 0
                eta = (len(orig_paths) - i - 1) / rate / 60 if rate > 0 else 0
                print(f"  [{i+1}/{len(orig_paths)}] {rate:.1f} img/s, ETA: {eta:.1f} min")

            try:
                res = future.result()
                records.extend(res)
            except Exception as e:
                print(f"Error processing image: {e}")

    elapsed = time.time() - t0
    print(f"  ✅ Done: {len(records)} measurements in {elapsed/60:.1f} min")
    return pd.DataFrame(records)


# ============================================================
# RUN ALL DATASETS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

os.makedirs(OUTPUT_DIR, exist_ok=True)

all_results = []

for name, cfg in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    if not os.path.isdir(cfg['base_dir']):
        print(f"  ❌ Base dir not found: {cfg['base_dir']}")
        print(f"  → Update DATASETS['{name}']['base_dir'] and rerun")
        continue

    df = compute_rd_metrics(
        base_dir=cfg['base_dir'],
        original_folder=cfg['original_folder'],
        cr_folders=cfg['cr_folders'],
        dataset_name=name,
        max_images=MAX_IMAGES_PER_DATASET,
    )

    if not df.empty:
        df.to_csv(os.path.join(OUTPUT_DIR, f'{name}_compression_metrics.csv'), index=False)
        all_results.append(df)

# ============================================================
# COMBINE WITH BREAKHIS & GENERATE PAPER TABLE
# ============================================================

BREAKHIS_CSV = '/content/drive/MyDrive/PATH_TO/compression_metrics.csv'  # ← UPDATE THIS

if os.path.exists(BREAKHIS_CSV):
    breakhis_df = pd.read_csv(BREAKHIS_CSV)
    breakhis_df = breakhis_df[['filename', 'compression_ratio', 'achieved_cr',
                                'compressed_bytes', 'bpp', 'psnr_db', 'ssim']].copy()
    breakhis_df['dataset'] = 'BreakHis'
    all_results.insert(0, breakhis_df)
    print(f"\n✅ Loaded BreakHis: {len(breakhis_df)} rows")
else:
    print(f"\n⚠️ BreakHis CSV not found at: {BREAKHIS_CSV}")
    print("  → Update BREAKHIS_CSV path")

if not all_results:
    print("\n❌ No results to summarize!")
else:
    combined = pd.concat(all_results, ignore_index=True)

    dataset_order = ['BreakHis', 'BACH', 'MHIST', 'NCT-CRC']
    cr_order = [15, 25, 50, 100, 500, 1000]

    summary_rows = []
    for ds in dataset_order:
        subset = combined[combined['dataset'] == ds]
        if subset.empty:
            continue
        for cr in cr_order:
            cr_data = subset[subset['compression_ratio'] == cr]
            if cr_data.empty:
                continue
            summary_rows.append({
                'Dataset': ds,
                'CR': f'CR{cr}',
                'N': len(cr_data),
                'Achieved CR': f"{cr_data['achieved_cr'].mean():.1f} ± {cr_data['achieved_cr'].std():.1f}",
                'PSNR (dB)': f"{cr_data['psnr_db'].mean():.2f} ± {cr_data['psnr_db'].std():.2f}",
                'SSIM': f"{cr_data['ssim'].mean():.4f} ± {cr_data['ssim'].std():.4f}",
                'psnr_mean': cr_data['psnr_db'].mean(),
                'ssim_mean': cr_data['ssim'].mean(),
            })

    summary = pd.DataFrame(summary_rows)

    print(f"\n{'='*80}")
    print("  RATE-DISTORTION SUMMARY FOR PAPER")
    print(f"{'='*80}")
    print(summary[['Dataset', 'CR', 'N', 'Achieved CR', 'PSNR (dB)', 'SSIM']].to_string(index=False))

    print(f"\n📊 QUALITY FLAGS:")
    for _, row in summary.iterrows():
        flags = []
        if row['psnr_mean'] < 30:
            flags.append('⚠️ PSNR < 30 dB')
        if row['ssim_mean'] < 0.7:
            flags.append('⚠️ SSIM < 0.7')
        if flags:
            print(f"  {row['Dataset']:>8s} {row['CR']:>6s}: {', '.join(flags)}")

    summary_path = os.path.join(OUTPUT_DIR, 'rate_distortion_summary_all.csv')
    summary.to_csv(summary_path, index=False)
    print(f"\n📄 Summary saved: {summary_path}")

    tex_lines = [
        r'\begin{table}[htbp]',
        r'\centering',
        r'\caption{Rate-distortion validation: PSNR and SSIM at each compression ratio.',
        r'Values reported as mean $\pm$ SD across all images.}',
        r'\label{tab:rate_distortion}',
        r'\begin{tabular}{llrccc}',
        r'\hline',
        r'Dataset & CR & N & Achieved CR & PSNR (dB) & SSIM \\',
        r'\hline',
    ]

    for ds in dataset_order:
        ds_rows = summary[summary['Dataset'] == ds]
        if ds_rows.empty:
            continue
        for i, (_, row) in enumerate(ds_rows.iterrows()):
            prefix = f"\\multirow{{{len(ds_rows)}}}{{*}}{{{ds}}}" if i == 0 else ""
            tex_lines.append(
                f"{prefix} & {row['CR']} & {row['N']} & "
                f"${row['Achieved CR'].replace('±', r'\pm')}$ & "
                f"${row['PSNR (dB)'].replace('±', r'\pm')}$ & "
                f"${row['SSIM'].replace('±', r'\pm')}$ \\\\"
            )
        tex_lines.append(r'\hline')

    tex_lines += [r'\end{tabular}', r'\end{table}']

    tex_path = os.path.join(OUTPUT_DIR, 'rate_distortion_table.tex')
    with open(tex_path, 'w') as f:
        f.write('\n'.join(tex_lines))
    print(f"📄 LaTeX table saved: {tex_path}")

    print(f"\n✅ ALL DONE. {len(combined)} total measurements across {combined['dataset'].nunique()} datasets.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  MHIST
  Found 3152 original images in /content/drive/MyDrive/MHIST_PLOS_ONE/MHIST_Split/CR1_original
  [1/3152] 2.3 img/s, ETA: 22.9 min
  [100/3152] 4.9 img/s, ETA: 10.3 min
  [200/3152] 2.6 img/s, ETA: 18.7 min
  [300/3152] 1.5 img/s, ETA: 31.6 min
  [400/3152] 1.2 img/s, ETA: 36.9 min
  [500/3152] 1.1 img/s, ETA: 39.2 min
  [600/3152] 1.1 img/s, ETA: 40.0 min
  [700/3152] 1.0 img/s, ETA: 41.3 min
  [800/3152] 0.9 img/s, ETA: 41.6 min
  [900/3152] 0.9 img/s, ETA: 41.3 min
  [1000/3152] 0.9 img/s, ETA: 41.4 min
  [1100/3152] 0.9 img/s, ETA: 40.2 min
  [1200/3152] 0.8 img/s, ETA: 38.8 min
  [1300/3152] 0.8 img/s, ETA: 37.3 min
  [1400/3152] 0.8 img/s, ETA: 35.7 min
  [1500/3152] 0.8 img/s, ETA: 34.0 min
  [1600/3152] 0.8 img/s, ETA: 32.3 min
  [1700/3152] 0.8 img/s, ETA: 30.4 min
  [1800/3152] 0.8 img/s, ETA: 28.5 min
  [1900/3152] 0.8 img/s, ETA: 26.6 min

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
/content/drive/MyDrive/BACH_PLOS_ONE
/content/drive/MyDrive/BreakHis Original
/content/drive/MyDrive/BreakHis_Analysis
/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL
/content/drive/MyDrive/BreakHis_PLOS_ONE_Results
/content/drive/MyDrive/BreakHis_PLOS_ONE_v2
/content/drive/MyDrive/BreakHis_PLOS_ONE_v3
/content/drive/MyDrive/BreakHis_PLOS_ONE_v4
/content/drive/MyDrive/BreakHis_PLOS_ONE_v5
/content/drive/MyDrive/BreakHis_Results
/content/drive/MyDrive/BreakHis_Split
/content/drive/MyDrive/JPEG2000-Cancer-Histopathology
/content/drive/MyDrive/JPEG2000_Paper_Final
/content/drive/MyDrive/NCT_CRC_PLOS_ONE


NameError: name 'content' is not defined

In [ ]:
"""
Rate-Distortion Metrics — FINAL VERSION (MULTIPROCESSING) - OPTIMIZED
=========================================================
- Pre-indexes files to bypass slow Google Drive I/O searches
- NCT-CRC: runs with 1000 image sample
- MHIST & BACH: loads from previous run CSVs
- BreakHis: loads from existing CSV
"""

# !pip install scikit-image --quiet

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import time
import warnings
import concurrent.futures
from functools import partial
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# CONFIGURATION — UPDATE ALL PATHS
# ============================================================

# --- Only NCT-CRC needs to run ---
NCT_CRC_CONFIG = {
    'base_dir': '/content/drive/MyDrive/NCT_CRC_PLOS_ONE/NCT_CRC_Split',
    'original_folder': 'CR1_original',
    'cr_folders': ['CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
}
NCT_CRC_MAX_IMAGES = 1000  # Sample 1000 out of 36K

# --- Already computed CSVs (from previous runs) ---
MHIST_CSV = '/content/drive/MyDrive/JPEG2000_Paper_Final/rate_distortion/MHIST_compression_metrics.csv'
BACH_CSV = '/content/drive/MyDrive/JPEG2000_Paper_Final/rate_distortion/BACH_compression_metrics.csv'
BREAKHIS_CSV = '/content/drive/MyDrive/JPEG2000_Paper_Final/rate_distortion/compression_metrics.csv'

OUTPUT_DIR = '/content/drive/MyDrive/JPEG2000_Paper_Final/rate_distortion'
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# CORE FUNCTION (MULTIPROCESSING)
# ============================================================

def _process_single_image(orig_path, orig_dir, base_dir, cr_folders, dataset_name, file_index):
    records = []
    try:
        orig_img = np.array(Image.open(orig_path).convert('RGB'))
    except Exception:
        return records

    rel_path = os.path.relpath(orig_path, orig_dir)
    filename = os.path.basename(orig_path)
    orig_size = os.path.getsize(orig_path)
    h, w = orig_img.shape[:2]
    stem = os.path.splitext(filename)[0]

    for cr_folder in cr_folders:
        cr_value = int(cr_folder.replace('CR', ''))

        # Use fast index lookup directly, bypassing os.path.exists
        found_path = file_index.get(cr_folder, {}).get(filename)
        if not found_path:
            # Fallback by stem in index
            found_path = file_index.get(cr_folder, {}).get(stem)

        if found_path:
            compressed_path = found_path
        else:
            continue

        try:
            comp_img = np.array(Image.open(compressed_path).convert('RGB'))
        except Exception:
            continue

        # Ensure same dimensions
        if orig_img.shape != comp_img.shape:
            comp_img = np.array(
                Image.fromarray(comp_img).resize(
                    (orig_img.shape[1], orig_img.shape[0]), Image.LANCZOS
                )
            )

        # Check if identical (codec floor)
        if np.array_equal(orig_img, comp_img):
            psnr = float('inf')
            ssim = 1.0
        else:
            psnr = compute_psnr(orig_img, comp_img, data_range=255)
            ssim = compute_ssim(
                orig_img, comp_img,
                data_range=255,
                channel_axis=2,
                win_size=min(7, min(orig_img.shape[0], orig_img.shape[1]))
            )

        comp_size = os.path.getsize(compressed_path)
        achieved_cr = orig_size / comp_size if comp_size > 0 else 0
        bpp = (comp_size * 8) / (h * w) if (h * w) > 0 else 0

        records.append({
            'dataset': dataset_name,
            'filename': filename,
            'compression_ratio': cr_value,
            'achieved_cr': round(achieved_cr, 4),
            'compressed_bytes': comp_size,
            'bpp': round(bpp, 6),
            'psnr_db': round(psnr, 4) if psnr != float('inf') else 'inf',
            'ssim': round(ssim, 6),
            'is_identical': psnr == float('inf'),
        })

    return records


def compute_rd_metrics(base_dir, original_folder, cr_folders, dataset_name, max_images=None):
    orig_dir = os.path.join(base_dir, original_folder)

    if not os.path.isdir(orig_dir):
        print(f"  ❌ Original folder not found: {orig_dir}")
        return pd.DataFrame()

    extensions = ('*.png', '*.PNG', '*.jpg', '*.JPG', '*.jpeg', '*.tif', '*.tiff')
    orig_paths = []
    for ext in extensions:
        orig_paths.extend(glob.glob(os.path.join(orig_dir, '**', ext), recursive=True))
    orig_paths = sorted(set(orig_paths))

    print(f"  Found {len(orig_paths)} original images in {orig_dir}")

    if max_images and len(orig_paths) > max_images:
        np.random.seed(42)
        idx = np.random.choice(len(orig_paths), max_images, replace=False)
        orig_paths = [orig_paths[i] for i in sorted(idx)]
        print(f"  Sampled {max_images} images")

    # --- OPTIMIZATION: Build Fast File Index ---
    print(f"  ⚡ Building fast file index to bypass Drive search...")
    file_index = {cr: {} for cr in cr_folders}
    for cr in cr_folders:
        cr_dir = os.path.join(base_dir, cr)
        if os.path.exists(cr_dir):
            for root, _, files in os.walk(cr_dir):
                for f in files:
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
                        full_path = os.path.join(root, f)
                        file_index[cr][f] = full_path
                        file_index[cr][os.path.splitext(f)[0]] = full_path

    records = []
    t0 = time.time()

    worker = partial(_process_single_image, orig_dir=orig_dir, base_dir=base_dir, cr_folders=cr_folders, dataset_name=dataset_name, file_index=file_index)

    # Using max_workers=os.cpu_count() + 2 to keep CPU busy while waiting for some I/O
    with concurrent.futures.ProcessPoolExecutor(max_workers=os.cpu_count() + 2) as executor:
        futures = {executor.submit(worker, path): path for path in orig_paths}

        for i, future in enumerate(concurrent.futures.as_completed(futures)):
            if (i + 1) % 50 == 0 or i == 0:
                elapsed = time.time() - t0
                rate = (i + 1) / elapsed if elapsed > 0 else 0
                eta = (len(orig_paths) - i - 1) / rate / 60 if rate > 0 else 0
                print(f"  [{i+1}/{len(orig_paths)}] {rate:.1f} img/s, ETA: {eta:.1f} min")

            try:
                res = future.result()
                records.extend(res)
            except Exception as e:
                print(f"  ⚠️ Error processing image: {e}")

    elapsed = time.time() - t0
    print(f"  ✅ Done: {len(records)} measurements in {elapsed/60:.1f} min")

    df = pd.DataFrame(records)
    if not df.empty and 'is_identical' in df.columns:
        for cr in sorted(df['compression_ratio'].unique()):
            cr_data = df[df['compression_ratio'] == cr]
            n_identical = cr_data['is_identical'].sum()
            if n_identical > 0:
                pct = n_identical / len(cr_data) * 100
                print(f"  ⚠️ CR{cr}: {n_identical}/{len(cr_data)} ({pct:.0f}%) images IDENTICAL to original (codec floor)")

    return df


# ============================================================
# RUN AND SUMMARIZE
# ============================================================

print("=" * 60)
print("  STEP 1: Computing NCT-CRC rate-distortion metrics")
print("=" * 60)

nct_df = compute_rd_metrics(
    base_dir=NCT_CRC_CONFIG['base_dir'],
    original_folder=NCT_CRC_CONFIG['original_folder'],
    cr_folders=NCT_CRC_CONFIG['cr_folders'],
    dataset_name='NCT-CRC',
    max_images=NCT_CRC_MAX_IMAGES,
)

if not nct_df.empty:
    nct_path = os.path.join(OUTPUT_DIR, 'NCT-CRC_compression_metrics.csv')
    nct_df.to_csv(nct_path, index=False)
    print(f"  📄 Saved: {nct_path}")

all_results = []
if not nct_df.empty:
    all_results.append(nct_df)
if os.path.exists(MHIST_CSV):
    all_results.append(pd.read_csv(MHIST_CSV).assign(dataset='MHIST', is_identical=lambda d: d['psnr_db'].astype(str) == 'inf'))
if os.path.exists(BACH_CSV):
    all_results.append(pd.read_csv(BACH_CSV).assign(dataset='BACH', is_identical=lambda d: d['psnr_db'].astype(str) == 'inf'))
if os.path.exists(BREAKHIS_CSV):
    all_results.append(pd.read_csv(BREAKHIS_CSV).assign(dataset='BreakHis', is_identical=False))

if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    print(f"\n✅ Fully compiled {len(combined)} records ready for Table.")


Mounted at /content/drive
  STEP 1: Computing NCT-CRC rate-distortion metrics


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
JPEG2000 Paper — Complete Project Scanner
==========================================
Scans all project folders, catalogs everything, maps to paper needs,
identifies gaps. Run on CPU in Colab.
"""

import os
import glob
import json
import csv
from collections import defaultdict
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# ALL PROJECT FOLDERS
# ============================================================

PROJECT_FOLDERS = [
    '/content/drive/MyDrive/BACH_PLOS_ONE',
    '/content/drive/MyDrive/BreakHis Original',
    '/content/drive/MyDrive/BreakHis_Analysis',
    '/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL',
    '/content/drive/MyDrive/BreakHis_PLOS_ONE_Results',
    '/content/drive/MyDrive/BreakHis_PLOS_ONE_v2',
    '/content/drive/MyDrive/BreakHis_PLOS_ONE_v3',
    '/content/drive/MyDrive/BreakHis_PLOS_ONE_v4',
    '/content/drive/MyDrive/BreakHis_PLOS_ONE_v5',
    '/content/drive/MyDrive/BreakHis_Results',
    '/content/drive/MyDrive/BreakHis_Split',
    '/content/drive/MyDrive/BreakHis_Compressed',
    '/content/drive/MyDrive/JPEG2000-Cancer-Histopathology',
    '/content/drive/MyDrive/JPEG2000_Paper_Final',
    '/content/drive/MyDrive/NCT_CRC_PLOS_ONE',
    '/content/drive/MyDrive/MHIST_PLOS_ONE',  # dropped but scan anyway
]

OUTPUT_FILE = '/content/drive/MyDrive/JPEG2000_Paper_Final/PROJECT_INVENTORY.txt'

# ============================================================
# SCANNER
# ============================================================

def get_file_info(filepath):
    """Get file size and modification time."""
    try:
        stat = os.stat(filepath)
        size_kb = stat.st_size / 1024
        mtime = datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d')
        return size_kb, mtime
    except:
        return 0, 'unknown'


def peek_csv(filepath, max_rows=5):
    """Read first few rows of a CSV to understand its contents."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            reader = csv.reader(f)
            rows = []
            for i, row in enumerate(reader):
                if i >= max_rows + 1:
                    break
                rows.append(row)

            # Count total rows
            f.seek(0)
            total_rows = sum(1 for _ in f) - 1  # minus header

        header = rows[0] if rows else []
        return header, total_rows
    except:
        return [], 0


def scan_folder(folder_path):
    """Scan a folder and categorize all files."""
    if not os.path.isdir(folder_path):
        return None

    inventory = {
        'path': folder_path,
        'name': os.path.basename(folder_path),
        'csvs': [],
        'notebooks': [],
        'models': [],
        'images': [],
        'figures': [],
        'code': [],
        'json_files': [],
        'tex_files': [],
        'other': [],
        'subfolders': [],
        'image_folders': {},  # CR folders with image counts
    }

    for root, dirs, files in os.walk(folder_path):
        # Track depth to avoid going too deep
        depth = root.replace(folder_path, '').count(os.sep)
        rel_root = os.path.relpath(root, folder_path)

        # Check for CR compression folders
        for d in dirs:
            if d.startswith('CR') or d == 'CR1_original':
                cr_path = os.path.join(root, d)
                img_count = 0
                for ext in ['*.png', '*.jpg', '*.PNG', '*.JPG']:
                    img_count += len(glob.glob(os.path.join(cr_path, '**', ext), recursive=True))
                if img_count > 0:
                    inventory['image_folders'][os.path.relpath(cr_path, folder_path)] = img_count

        # Track subfolders at depth 0 and 1
        if depth <= 1:
            for d in sorted(dirs):
                if not d.startswith('.'):
                    inventory['subfolders'].append(os.path.relpath(os.path.join(root, d), folder_path))

        for f in files:
            if f.startswith('.'):
                continue

            filepath = os.path.join(root, f)
            size_kb, mtime = get_file_info(filepath)
            rel_path = os.path.relpath(filepath, folder_path)
            ext = os.path.splitext(f)[1].lower()

            entry = {
                'name': f,
                'path': rel_path,
                'size_kb': round(size_kb, 1),
                'modified': mtime,
            }

            if ext == '.csv':
                header, n_rows = peek_csv(filepath)
                entry['columns'] = header[:15]  # first 15 columns
                entry['n_rows'] = n_rows
                inventory['csvs'].append(entry)

            elif ext == '.ipynb':
                inventory['notebooks'].append(entry)

            elif ext in ['.h5', '.hdf5', '.keras', '.pb', '.pth', '.pt', '.ckpt']:
                inventory['models'].append(entry)

            elif ext in ['.png', '.jpg', '.jpeg', '.pdf'] and depth <= 2:
                # Only track figures/plots, not dataset images
                if any(kw in rel_path.lower() for kw in ['fig', 'plot', 'chart', 'heatmap',
                        'comparison', 'diagnostic', 'analysis', 'model_selection']):
                    inventory['figures'].append(entry)
                elif 'results' in rel_path.lower() or 'figures' in rel_path.lower():
                    inventory['figures'].append(entry)

            elif ext == '.py':
                inventory['code'].append(entry)

            elif ext == '.json':
                # Try to read key info
                try:
                    with open(filepath, 'r') as jf:
                        data = json.load(jf)
                    entry['keys'] = list(data.keys())[:10] if isinstance(data, dict) else f'list[{len(data)}]'
                except:
                    entry['keys'] = 'unreadable'
                inventory['json_files'].append(entry)

            elif ext == '.tex':
                inventory['tex_files'].append(entry)

            elif ext == '.txt':
                inventory['other'].append(entry)

    return inventory


def identify_experiment_results(csv_entry):
    """Identify what kind of experiment data a CSV contains."""
    name = csv_entry['name'].lower()
    cols = [c.lower() for c in csv_entry.get('columns', [])]
    n_rows = csv_entry.get('n_rows', 0)

    tags = []

    # Experiment type
    if 'adaptation' in name or 'experiment1' in name:
        tags.append('EXP1_ADAPTATION')
    if 'robustness' in name or 'experiment2' in name:
        tags.append('EXP2_ROBUSTNESS')
    if 'compression_aware' in name or 'experiment3' in name or 'cat' in name:
        tags.append('EXP3_CAT')

    # Data type
    if 'model_selection' in name or 'quick_test' in name or 'hybrid_test' in name or 'texture_hybrid' in name:
        tags.append('MODEL_SELECTION')
    if 'sustainability' in name or 'co2' in name or 'emission' in name:
        tags.append('SUSTAINABILITY')
    if 'compression_metric' in name or 'rate_distortion' in name:
        tags.append('RATE_DISTORTION')
    if 'feature' in name or 'texture' in name or 'frequency' in name:
        tags.append('FEATURE_ANALYSIS')
    if 'statistical' in name or 'significance' in name:
        tags.append('STATISTICS')

    # Check columns for clues
    col_str = ' '.join(cols)
    if 'psnr' in col_str or 'ssim' in col_str:
        tags.append('HAS_PSNR_SSIM')
    if 'accuracy' in col_str or 'f1' in col_str or 'auc' in col_str:
        tags.append('HAS_CLASSIFICATION_METRICS')
    if 'fold' in col_str:
        tags.append('HAS_CV_FOLDS')
    if 'co2' in col_str or 'emission' in col_str:
        tags.append('HAS_EMISSIONS')

    # Dataset
    path_lower = csv_entry.get('path', '').lower()
    if 'breakhis' in path_lower or 'breakhis' in name:
        tags.append('DATASET:BREAKHIS')
    if 'bach' in path_lower or 'bach' in name:
        tags.append('DATASET:BACH')
    if 'mhist' in path_lower or 'mhist' in name:
        tags.append('DATASET:MHIST')
    if 'nct' in path_lower or 'nct' in name:
        tags.append('DATASET:NCT-CRC')

    return tags


# ============================================================
# MAIN SCAN
# ============================================================

print("=" * 70)
print("  JPEG2000 PAPER — COMPLETE PROJECT INVENTORY")
print("=" * 70)

all_inventories = {}
all_csvs = []

for folder in PROJECT_FOLDERS:
    inv = scan_folder(folder)
    if inv:
        all_inventories[folder] = inv
        for csv_entry in inv['csvs']:
            csv_entry['source_folder'] = inv['name']
            csv_entry['tags'] = identify_experiment_results(csv_entry)
            all_csvs.append(csv_entry)

# ============================================================
# PRINT REPORT
# ============================================================

report_lines = []

def p(line=''):
    print(line)
    report_lines.append(line)

p("=" * 70)
p("  SECTION 1: FOLDER-BY-FOLDER INVENTORY")
p("=" * 70)

for folder, inv in all_inventories.items():
    p(f"\n{'#' * 60}")
    p(f"  📁 {inv['name']}")
    p(f"  {folder}")
    p(f"{'#' * 60}")

    if inv['subfolders']:
        p(f"\n  Subfolders: {', '.join(inv['subfolders'][:20])}")

    if inv['image_folders']:
        p(f"\n  🖼️  COMPRESSION IMAGE FOLDERS:")
        for cr_path, count in sorted(inv['image_folders'].items()):
            p(f"    {cr_path}: {count} images")

    if inv['csvs']:
        p(f"\n  📊 CSV FILES ({len(inv['csvs'])}):")
        for csv_entry in inv['csvs']:
            tags = identify_experiment_results(csv_entry)
            tag_str = f" [{', '.join(tags)}]" if tags else ""
            cols_preview = ', '.join(csv_entry.get('columns', [])[:6])
            p(f"    {csv_entry['path']}")
            p(f"      {csv_entry['n_rows']} rows | {csv_entry['size_kb']}KB | {csv_entry['modified']}")
            p(f"      Columns: {cols_preview}...")
            p(f"      Tags: {tag_str}")

    if inv['notebooks']:
        p(f"\n  📓 NOTEBOOKS ({len(inv['notebooks'])}):")
        for nb in inv['notebooks']:
            p(f"    {nb['path']} ({nb['size_kb']:.0f}KB, {nb['modified']})")

    if inv['models']:
        p(f"\n  🧠 SAVED MODELS ({len(inv['models'])}):")
        for m in inv['models']:
            p(f"    {m['path']} ({m['size_kb']/1024:.1f}MB, {m['modified']})")

    if inv['figures']:
        p(f"\n  📈 FIGURES ({len(inv['figures'])}):")
        for fig in inv['figures']:
            p(f"    {fig['path']} ({fig['size_kb']:.0f}KB)")

    if inv['json_files']:
        p(f"\n  📄 JSON FILES ({len(inv['json_files'])}):")
        for jf in inv['json_files']:
            keys_str = ', '.join(jf['keys'][:5]) if isinstance(jf['keys'], list) else str(jf['keys'])
            p(f"    {jf['path']} — keys: {keys_str}")

    if inv['tex_files']:
        p(f"\n  📝 TEX FILES ({len(inv['tex_files'])}):")
        for tf in inv['tex_files']:
            p(f"    {tf['path']}")


# ============================================================
# SECTION 2: WHAT THE PAPER NEEDS vs WHAT EXISTS
# ============================================================

p(f"\n\n{'=' * 70}")
p("  SECTION 2: PAPER REQUIREMENTS vs EXISTING DATA")
p("=" * 70)

requirements = {
    'BREAKHIS_EXP1': {
        'desc': 'BreakHis Experiment 1 (Adaptation)',
        'tags': ['EXP1_ADAPTATION', 'DATASET:BREAKHIS'],
        'needed': True,
    },
    'BREAKHIS_EXP2': {
        'desc': 'BreakHis Experiment 2 (Robustness)',
        'tags': ['EXP2_ROBUSTNESS', 'DATASET:BREAKHIS'],
        'needed': True,
    },
    'BREAKHIS_EXP3': {
        'desc': 'BreakHis Experiment 3 (CAT)',
        'tags': ['EXP3_CAT', 'DATASET:BREAKHIS'],
        'needed': True,
    },
    'BREAKHIS_RD': {
        'desc': 'BreakHis Rate-Distortion (PSNR/SSIM)',
        'tags': ['RATE_DISTORTION', 'DATASET:BREAKHIS'],
        'alt_tags': ['HAS_PSNR_SSIM', 'DATASET:BREAKHIS'],
        'needed': True,
    },
    'BREAKHIS_STATS': {
        'desc': 'BreakHis Statistical Analysis',
        'tags': ['STATISTICS', 'DATASET:BREAKHIS'],
        'needed': True,
    },
    'BREAKHIS_MODEL_SEL': {
        'desc': 'BreakHis Model Selection (13 architectures)',
        'tags': ['MODEL_SELECTION'],
        'needed': True,
    },
    'BREAKHIS_SUSTAINABILITY': {
        'desc': 'BreakHis Sustainability/CO2 Data',
        'tags': ['SUSTAINABILITY', 'DATASET:BREAKHIS'],
        'alt_tags': ['HAS_EMISSIONS'],
        'needed': True,
    },
    'BREAKHIS_FEATURES': {
        'desc': 'BreakHis Feature Analysis (texture, frequency)',
        'tags': ['FEATURE_ANALYSIS'],
        'needed': True,
    },
    'BREAKHIS_COMPRESSED_IMAGES': {
        'desc': 'BreakHis Compressed Images (CR1–CR1000)',
        'tags': ['COMPRESSED_IMAGES_BREAKHIS'],
        'needed': True,
    },
    'BACH_EXP1': {
        'desc': 'BACH Experiment 1 (Adaptation)',
        'tags': ['EXP1_ADAPTATION', 'DATASET:BACH'],
        'needed': True,
    },
    'BACH_EXP2': {
        'desc': 'BACH Experiment 2 (Robustness)',
        'tags': ['EXP2_ROBUSTNESS', 'DATASET:BACH'],
        'needed': True,
    },
    'BACH_EXP3': {
        'desc': 'BACH Experiment 3 (CAT)',
        'tags': ['EXP3_CAT', 'DATASET:BACH'],
        'needed': True,
    },
    'BACH_RD': {
        'desc': 'BACH Rate-Distortion (PSNR/SSIM)',
        'tags': ['RATE_DISTORTION', 'DATASET:BACH'],
        'needed': True,
    },
    'BACH_SUSTAINABILITY': {
        'desc': 'BACH Sustainability/CO2 Data',
        'tags': ['SUSTAINABILITY', 'DATASET:BACH'],
        'needed': True,
    },
    'NCT_EXP1': {
        'desc': 'NCT-CRC Experiment 1 (Adaptation)',
        'tags': ['EXP1_ADAPTATION', 'DATASET:NCT-CRC'],
        'needed': True,
    },
    'NCT_EXP2': {
        'desc': 'NCT-CRC Experiment 2 (Robustness)',
        'tags': ['EXP2_ROBUSTNESS', 'DATASET:NCT-CRC'],
        'needed': True,
    },
    'NCT_EXP3': {
        'desc': 'NCT-CRC Experiment 3 (CAT)',
        'tags': ['EXP3_CAT', 'DATASET:NCT-CRC'],
        'needed': True,
    },
    'NCT_RD': {
        'desc': 'NCT-CRC Rate-Distortion (PSNR/SSIM)',
        'tags': ['RATE_DISTORTION', 'DATASET:NCT-CRC'],
        'needed': True,
    },
    'NCT_SUSTAINABILITY': {
        'desc': 'NCT-CRC Sustainability/CO2 Data',
        'tags': ['SUSTAINABILITY', 'DATASET:NCT-CRC'],
        'needed': True,
    },
}

p(f"\n  {'Requirement':<45s} {'Status':<10s} {'Location'}")
p(f"  {'─' * 100}")

found_items = {}

for req_id, req in requirements.items():
    matches = []
    for csv_entry in all_csvs:
        csv_tags = csv_entry.get('tags', [])

        # Check primary tags
        if all(t in csv_tags for t in req['tags']):
            matches.append(csv_entry)
        # Check alternate tags
        elif 'alt_tags' in req and all(t in csv_tags for t in req['alt_tags']):
            matches.append(csv_entry)

    # Special check for compressed images
    if 'COMPRESSED_IMAGES' in req_id:
        for folder, inv in all_inventories.items():
            if inv['image_folders']:
                folder_name = inv['name'].lower()
                if 'breakhis' in req_id.lower() and 'breakhis' in folder_name:
                    matches.append({'name': f"Image folders in {inv['name']}", 'path': folder,
                                     'source_folder': inv['name']})
                elif 'bach' in req_id.lower() and 'bach' in folder_name:
                    matches.append({'name': f"Image folders in {inv['name']}", 'path': folder,
                                     'source_folder': inv['name']})
                elif 'nct' in req_id.lower() and 'nct' in folder_name:
                    matches.append({'name': f"Image folders in {inv['name']}", 'path': folder,
                                     'source_folder': inv['name']})

    if matches:
        status = '✅ FOUND'
        location = f"{matches[0]['source_folder']}/{matches[0]['name']}"
        if len(matches) > 1:
            location += f" (+{len(matches)-1} more)"
        found_items[req_id] = matches
    else:
        status = '❌ MISSING'
        location = ''

    p(f"  {req['desc']:<45s} {status:<10s} {location}")


# ============================================================
# SECTION 3: GAP ANALYSIS
# ============================================================

p(f"\n\n{'=' * 70}")
p("  SECTION 3: WHAT YOU NEED TO DO")
p("=" * 70)

missing = [req_id for req_id, req in requirements.items() if req_id not in found_items]
found = [req_id for req_id in requirements if req_id in found_items]

p(f"\n  ✅ FOUND: {len(found)}/{len(requirements)} requirements met")
p(f"  ❌ MISSING: {len(missing)}/{len(requirements)} requirements")

if missing:
    p(f"\n  MISSING ITEMS (action needed):")
    for req_id in missing:
        req = requirements[req_id]
        p(f"    ❌ {req['desc']}")

    # Group by action needed
    p(f"\n  RECOMMENDED ACTIONS:")

    nct_missing = [r for r in missing if r.startswith('NCT')]
    breakhis_missing = [r for r in missing if r.startswith('BREAKHIS')]
    bach_missing = [r for r in missing if r.startswith('BACH')]

    if nct_missing:
        p(f"\n  🔬 NCT-CRC EXPERIMENTS ({len(nct_missing)} items):")
        for r in nct_missing:
            p(f"     → {requirements[r]['desc']}")
        p(f"     ACTION: Run classification experiments (Exp 1, 2, 3) on NCT-CRC")
        p(f"     ESTIMATED TIME: ~4-6 hours on Colab GPU (L4)")

    if breakhis_missing:
        p(f"\n  🔬 BREAKHIS ({len(breakhis_missing)} items):")
        for r in breakhis_missing:
            p(f"     → {requirements[r]['desc']}")
        p(f"     ACTION: Locate existing data or re-run")

    if bach_missing:
        p(f"\n  🔬 BACH ({len(bach_missing)} items):")
        for r in bach_missing:
            p(f"     → {requirements[r]['desc']}")


# ============================================================
# SECTION 4: DUPLICATE / SCATTERED DATA
# ============================================================

p(f"\n\n{'=' * 70}")
p("  SECTION 4: SCATTERED / DUPLICATE DATA")
p("=" * 70)

# Find CSVs with similar names across folders
csv_by_name = defaultdict(list)
for csv_entry in all_csvs:
    csv_by_name[csv_entry['name']].append(csv_entry)

duplicates = {name: entries for name, entries in csv_by_name.items() if len(entries) > 1}

if duplicates:
    p(f"\n  ⚠️  FILES FOUND IN MULTIPLE LOCATIONS:")
    for name, entries in duplicates.items():
        p(f"\n    {name}:")
        for e in entries:
            p(f"      {e['source_folder']}/{e['path']} ({e['n_rows']} rows, {e['modified']})")
        # Recommend which to use (latest date, most rows)
        best = max(entries, key=lambda x: (x['n_rows'], x['modified']))
        p(f"      → RECOMMEND: {best['source_folder']}/{best['path']} (most rows, latest)")


# ============================================================
# SECTION 5: COMPLETE CSV CATALOG
# ============================================================

p(f"\n\n{'=' * 70}")
p("  SECTION 5: COMPLETE CSV CATALOG")
p("=" * 70)

for csv_entry in sorted(all_csvs, key=lambda x: x['source_folder']):
    tags = csv_entry.get('tags', [])
    tag_str = ', '.join(tags) if tags else 'UNTAGGED'
    p(f"\n  {csv_entry['source_folder']}/{csv_entry['path']}")
    p(f"    Rows: {csv_entry['n_rows']} | Size: {csv_entry['size_kb']}KB | Modified: {csv_entry['modified']}")
    p(f"    Columns: {', '.join(csv_entry.get('columns', [])[:8])}")
    p(f"    Tags: [{tag_str}]")


# ============================================================
# SECTION 6: BREAKHIS compression_metrics.csv FINDER
# ============================================================

p(f"\n\n{'=' * 70}")
p("  SECTION 6: SEARCHING FOR BREAKHIS compression_metrics.csv")
p("=" * 70)

found_breakhis_rd = []
for csv_entry in all_csvs:
    cols = [c.lower() for c in csv_entry.get('columns', [])]
    if ('psnr' in ' '.join(cols) or 'psnr_db' in ' '.join(cols)) and csv_entry['n_rows'] > 1000:
        found_breakhis_rd.append(csv_entry)
        p(f"\n  🔍 CANDIDATE: {csv_entry['source_folder']}/{csv_entry['path']}")
        p(f"     Rows: {csv_entry['n_rows']} | Columns: {', '.join(csv_entry.get('columns', [])[:10])}")

if not found_breakhis_rd:
    p(f"\n  ❌ No large CSV with PSNR/SSIM columns found across all folders.")
    p(f"     The BreakHis compression_metrics.csv may need to be regenerated.")


# ============================================================
# SAVE REPORT
# ============================================================

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
with open(OUTPUT_FILE, 'w') as f:
    f.write('\n'.join(report_lines))

p(f"\n\n{'=' * 70}")
p(f"  📄 FULL REPORT SAVED: {OUTPUT_FILE}")
p(f"  Share this file with Claude for targeted help.")
p(f"{'=' * 70}")

Mounted at /content/drive
  JPEG2000 PAPER — COMPLETE PROJECT INVENTORY
  SECTION 1: FOLDER-BY-FOLDER INVENTORY

############################################################
  📁 BACH_PLOS_ONE
  /content/drive/MyDrive/BACH_PLOS_ONE
############################################################

  Subfolders: BACH_PLOS_ONE_FINAL, BACH_PLOS_ONE_FINAL1, BACH_Split, BACH_analysis, BACH_Split/CR100, BACH_Split/CR1000, BACH_Split/CR15, BACH_Split/CR1_original, BACH_Split/CR25, BACH_Split/CR50, BACH_Split/CR500, BACH_analysis/figures, BACH_PLOS_ONE_FINAL/emissions, BACH_PLOS_ONE_FINAL/figures, BACH_PLOS_ONE_FINAL/models, BACH_PLOS_ONE_FINAL/results, BACH_PLOS_ONE_FINAL1/emissions, BACH_PLOS_ONE_FINAL1/figures, BACH_PLOS_ONE_FINAL1/models, BACH_PLOS_ONE_FINAL1/results

  🖼️  COMPRESSION IMAGE FOLDERS:
    BACH_Split/CR100: 400 images
    BACH_Split/CR1000: 400 images
    BACH_Split/CR15: 400 images
    BACH_Split/CR1_original: 400 images
    BACH_Split/CR25: 400 images
    BACH_Split/CR50: 400 im